In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/layoutlm/medquad.csv
/kaggle/input/layoutlm/models--yikuan8--Clinical-Longformer/refs/main
/kaggle/input/layoutlm/models--yikuan8--Clinical-Longformer/blobs/75801df688e89a96f642b98cbdd97288f5207518
/kaggle/input/layoutlm/models--yikuan8--Clinical-Longformer/blobs/dcbb04799764dccba6c5dfeef25f62e9955b7d07
/kaggle/input/layoutlm/models--yikuan8--Clinical-Longformer/blobs/87e239f555459c3668993fbf5b53bea93f503be86f2f4285f431fdb2c6c33c75
/kaggle/input/layoutlm/models--yikuan8--Clinical-Longformer/blobs/6636bda4a1fd7a63653dffb22683b8162c8de956
/kaggle/input/layoutlm/models--yikuan8--Clinical-Longformer/blobs/2ea7ad0e45a9d1d1591782ba7e29a703d0758831
/kaggle/input/layoutlm/models--yikuan8--Clinical-Longformer/blobs/49684d2d368a3a66b7056f79ebeb7d236a685f6d
/kaggle/input/layoutlm/models--yikuan8--Clinical-Longformer/blobs/4ebe4bb3f3114daf2e4cc349f24873a1175a35d7
/kaggle/input/layoutlm/models--yikuan8--Clinical-Longformer/.no_exist/dc05ee5437027609b953618bc8e2b725a30bd670/added_token

In [2]:
!nvidia-smi

Sat Nov 29 07:38:26 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.172.08             Driver Version: 570.172.08     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install --q qdrant-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.5/378.5 kB 7.6 MB/s eta 0:00:00a 0:00:01


In [4]:
import torch
import gradio as gr
import pandas as pd
import transformers
import qdrant_client
import sentence_transformers
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
import warnings
warnings.filterwarnings('ignore')


2025-11-29 07:38:49.223257: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764401929.388656      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764401929.433743      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [5]:
#For reproduction of the same results as here
set_seed(0)

In [6]:
%env TOKENIZERS_PARALLELISM=True

env: TOKENIZERS_PARALLELISM=True


In [7]:
# Load data into the RAG module
df = pd.read_csv('/kaggle/input/layoutlm/medquad.csv')

In [8]:
df

,question,answer,source,focus_area
0,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma
1,What causes Glaucoma ?,"Nearly 2.7 million people have glaucoma, a lea...",NIHSeniorHealth,Glaucoma
2,What are the symptoms of Glaucoma ?,Symptoms of Glaucoma Glaucoma can develop in ...,NIHSeniorHealth,Glaucoma
3,What are the treatments for Glaucoma ?,"Although open-angle glaucoma cannot be cured, ...",NIHSeniorHealth,Glaucoma
4,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma
...,...,...,...,...
16407,What is (are) Diabetic Neuropathies: The Nerve...,Focal neuropathy appears suddenly and affects ...,NIDDK,Diabetic Neuropathies: The Nerve Damage of Dia...
16408,How to prevent Diabetic Neuropathies: The Nerv...,The best way to prevent neuropathy is to keep ...,NIDDK,Diabetic Neuropathies: The Nerve Damage of Dia...
16409,How to diagnose Diabetic Neuropathies: The Ner...,Doctors diagnose neuropathy on the basis of sy...,NIDDK,Diabetic Neuropathies: The Nerve Damage of Dia...
16410,What are the treatments for Diabetic Neuropath...,The first treatment step is to bring blood glu...,NIDDK,Diabetic Neuropathies: The Nerve Damage of Dia...


In [9]:
ques_data = df['question'].tolist()[:20000]
answer_data = df['answer'].tolist()[:20000]

Embeddings Model
https://huggingface.co/sentence-transformers/all-mpnet-base-v2

In [10]:
# Defines the embeddings model
modelo_embedding = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
# Defines the embeddings 
modelo_embedding = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

RAG Module with Vector Database
https://qdrant.tech/

In [12]:
# Use the embeddings model to create vectors from the data
vetores = modelo_embedding.encode(ques_data)

In [13]:
# Create the client for the vector database defined in memory
banco_vetorial = QdrantClient(":memory:")

In [14]:
# Create the collection in the vector database
banco_vetorial.create_collection(collection_name = "doc_data",
                                 vectors_config = VectorParams(size = len(vetores[0]),
                                                               distance = Distance.COSINE))

True

In [15]:
# Upload data to the vector database
banco_vetorial.upload_collection(collection_name = "doc_data",
                                 ids = [i for i in range(len(ques_data))],
                                 vectors = vetores)

Information Retrieval Module¶


In [16]:
# Define the function that receives a question as input
def dsa_recupera_dados(question):

    # Loads the sentence transformer model "all-mpnet-base-v2" from the SentenceTransformer library
    model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

    # Encodes the question into an embedding vector using the model
    ques_vector = model.encode(question)

    # Queries the vector database with the question vector, searching for similar documents
    result = banco_vetorial.query_points(collection_name="doc_data", query=ques_vector)

    # Creates an empty list to store the IDs of the most similar documents
    sim_ids = []

    # Iterates over the query results and adds the document IDs to the list
    for i in result.points:
        sim_ids.append(i.id)

    # Retrieves the context of the most similar document based on the first ID in the list
    context = answer_data[sim_ids[0]]

    # Returns the document context as the answer
    return context

Integration Module Between SLM and RAG¶
https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0

In [17]:
# Defines the function 'dsa_llm_rag' that receives a question and a context as input
def dsa_llm_rag(question, context):

    # Defines the name of the language model to be used, in this case "TinyLlama-1.1B-Chat-v1.0"
    nome_llm = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

    # Loads the tokenizer associated with the model using the specified name
    tokenizer = AutoTokenizer.from_pretrained(nome_llm)

    # Loads the causal language model using the specified name
    # Replace "cuda" with "cpu" if running locally
    model = AutoModelForCausalLM.from_pretrained(nome_llm, device_map="cuda")

    # Defines the app prompt, which includes the question and the context,
    # instructing the LLM to respond based on the context
    chat = [{"role": "user", "content": f"this is question {question} asked by user you are a medical clinic assistant answer the question based on this context {context} in not more than 3-4 points"}]

    # Applies the chat template, tokenizes the prompt and converts it to PyTorch tensors,
    # adding the generation prompt
    # Remove the final [.to("cuda")] part if running locally
    token_inputs = tokenizer.apply_chat_template(chat,
                                                 tokenize=True,
                                                 return_tensors="pt",
                                                 add_generation_prompt=True).to("cuda")

    # Generates the model's response from the input tokens using sampling
    # and setting limits such as the maximum number of new tokens and temperature (creativity level)
    token_outputs = model.generate(input_ids=token_inputs,
                                   do_sample=True,
                                   max_new_tokens=500,
                                   temperature=1.5)

    # Extracts the new tokens generated that are not part of the original input
    new_tokens = token_outputs[0][token_inputs.shape[-1]:]

    # Decodes the new tokens into text, ignoring special tokens
    decoded_output = tokenizer.decode(new_tokens, skip_special_tokens=True)

    # Returns the decoded text as the answer
    return decoded_output


In [18]:
# Defines the function 'dsa_gera_resultado' that receives the user input as a parameter
def dsa_gera_resultado(user_input):

    # Checks if the user has provided an input
    if user_input:

        # Retrieves the relevant context by calling the 'dsa_recupera_dados' function with the user input
        context = dsa_recupera_dados(user_input)

        # Generates and returns the answer by calling the 'dsa_llm_rag' function with the user input and retrieved context
        return dsa_llm_rag(user_input, context)

    # If the user input is empty, returns a message asking to enter a question
    else:
        return "Please enter your question."


Web Application Module To Deploy

In [19]:
# Defining the custom interface
webapp = gr.Interface(

    # Function that processes the user input
    fn=dsa_gera_resultado,

    # Larger text box with a custom placeholder
    inputs=gr.Textbox(lines=2, placeholder="Type your question here...", label="Input"),

    # Output text box with a custom label
    outputs=gr.Textbox(label="Answer"),

    # Application title
    title="Deep Learning - RAG - Project",

    # Custom description
    description="This is an AI application for automating the medical patient triage process.",

    # Pre-defined examples
    examples=[["What is (are) Parasites - Schistosomiasis ?"]],

    # More compact theme
    theme="compact"
)

In [20]:
# Launch the Gradio interface
webapp.launch(share = True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://91741563ec79e61145.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
